# 如何格式化 ChatGPT 模型的输入

ChatGPT 由 OpenAI 最先进的模型 `gpt-3.5-turbo` 和 `gpt-4` 驱动。

你可以通过 OpenAI API，使用 `gpt-3.5-turbo` 或 `gpt-4` 构建自己的应用。

聊天模型接收一系列消息作为输入，并返回一条由 AI 撰写的消息作为输出。

本指南通过几个 API 调用示例来说明聊天格式。

## 1. 导入 openai 库

In [ ]:
# if needed, install and/or upgrade to the latest version of the OpenAI Python library
%pip install --upgrade openai


In [22]:
# import the OpenAI Python library for calling the OpenAI API
from openai import OpenAI
import os

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "<your OpenAI API key if not set as env var>"))

## 2. chat completion API 调用示例

chat completion API 调用的参数：
**必填**
- `model`：要使用的模型名称（例如 `gpt-3.5-turbo`、`gpt-4`、`gpt-3.5-turbo-16k-1106`）
- `messages`：消息对象列表，每个对象有两个必填字段：
    - `role`：发送者的角色（`system`、`user`、`assistant` 或 `tool`）
    - `content`：消息内容（例如 `Write me a beautiful poem`）

消息还可以包含一个可选的 `name` 字段，用于为发送者命名，例如 `example-user`、`Alice`、`BlackbeardBot`。名称中不能包含空格。

**可选**
- `frequency_penalty`：根据 token 出现的频率进行惩罚，减少重复。
- `logit_bias`：通过偏置值修改指定 token 的出现概率。
- `logprobs`：为 true 时返回输出 token 的对数概率。
- `top_logprobs`：指定在每个位置返回的最可能 token 的数量。
- `max_tokens`：设置 chat completion 中生成 token 的最大数量。
- `n`：为每个输入生成指定数量的 chat completion 选项。
- `presence_penalty`：根据 token 是否已在文本中出现，对新 token 进行惩罚。
- `response_format`：指定输出格式，例如 JSON 模式。
- `seed`：使用指定的种子确保确定性采样。
- `stop`：指定最多 4 个序列，API 遇到时将停止生成 token。
- `stream`：在 token 可用时发送部分消息增量。
- `temperature`：设置采样温度，取值在 0 到 2 之间。
- `top_p`：使用核采样；只考虑累积概率质量为 top_p 的 token。
- `tools`：列出模型可以调用的函数。
- `tool_choice`：控制模型的函数调用（none/auto/function）。
- `user`：终端用户的唯一标识符，用于监控和滥用检测。


自 2024 年 1 月起，你还可以选择提交一个 `functions` 列表，告诉 GPT 是否可以生成 JSON 并传给某个函数。详情请参阅[文档](https://platform.openai.com/docs/guides/function-calling)、[API 参考](https://platform.openai.com/docs/api-reference/chat)，或 Cookbook 指南[如何使用聊天模型调用函数](How_to_call_functions_with_chat_models.ipynb)。

通常，一段对话以一条告诉助手如何表现的 system 消息开始，之后是交替出现的 user 和 assistant 消息，但你不必遵循这种格式。

我们来看一个聊天 API 调用示例，了解聊天格式在实践中如何运作。

In [2]:
# Example OpenAI Python library request
MODEL = "gpt-3.5-turbo"
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Knock knock."},
        {"role": "assistant", "content": "Who's there?"},
        {"role": "user", "content": "Orange."},
    ],
    temperature=0,
)


In [7]:
print(json.dumps(json.loads(response.model_dump_json()), indent=4))

{
    "id": "chatcmpl-8dee9DuEFcg2QILtT2a6EBXZnpirM",
    "choices": [
        {
            "finish_reason": "stop",
            "index": 0,
            "logprobs": null,
            "message": {
                "content": "Orange who?",
                "role": "assistant",
                "function_call": null,
                "tool_calls": null
            }
        }
    ],
    "created": 1704461729,
    "model": "gpt-3.5-turbo-0613",
    "object": "chat.completion",
    "system_fingerprint": null,
    "usage": {
        "completion_tokens": 3,
        "prompt_tokens": 35,
        "total_tokens": 38
    }
}


可以看到，响应对象包含以下几个字段：
- `id`：请求的 ID
- `choices`：completion 对象列表（只有一个，除非将 `n` 设为大于 1）
    - `finish_reason`：模型停止生成文本的原因（`stop`，或在达到 `max_tokens` 限制时为 `length`）
    - `index`：该选项在选项列表中的索引。
    - `logprobs`：该选项的对数概率信息。
    - `message`：模型生成的消息对象
        - `content`：消息内容
        - `role`：该消息作者的角色。
        - `tool_calls`：模型生成的工具调用，例如函数调用（如果提供了 tools）
- `created`：请求的时间戳
- `model`：用于生成响应的模型全名
- `object`：返回对象的类型（例如 `chat.completion`）
- `system_fingerprint`：该指纹代表模型运行时所用的后端配置。
- `usage`：生成回复所用的 token 数量，分别统计 prompt、completion 和总数

仅提取回复：

In [9]:
response.choices[0].message.content


'Orange who?'

即使是非对话类任务，也可以通过把指令放在第一条 user 消息中来套用聊天格式。

例如，要让模型以海盗黑胡子的风格解释异步编程，可以这样构造对话：

In [10]:
# example with a system message
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain asynchronous programming in the style of the pirate Blackbeard."},
    ],
    temperature=0,
)

print(response.choices[0].message.content)


Arr, me matey! Let me tell ye a tale of asynchronous programming, in the style of the fearsome pirate Blackbeard!

Picture this, me hearties. In the vast ocean of programming, there be times when ye need to perform multiple tasks at once. But fear not, for asynchronous programming be here to save the day!

Ye see, in traditional programming, ye be waitin' for one task to be done before movin' on to the next. But with asynchronous programming, ye can be takin' care of multiple tasks at the same time, just like a pirate multitaskin' on the high seas!

Instead of waitin' for a task to be completed, ye can be sendin' it off on its own journey, while ye move on to the next task. It be like havin' a crew of trusty sailors, each takin' care of their own duties, without waitin' for the others.

Now, ye may be wonderin', how does this sorcery work? Well, me matey, it be all about callbacks and promises. When ye be sendin' off a task, ye be attachin' a callback function to it. This be like leavi

In [11]:
# example without a system message
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Explain asynchronous programming in the style of the pirate Blackbeard."},
    ],
    temperature=0,
)

print(response.choices[0].message.content)


Arr, me hearties! Gather 'round and listen up, for I be tellin' ye about the mysterious art of asynchronous programming, in the style of the fearsome pirate Blackbeard!

Now, ye see, in the world of programming, there be times when we need to perform tasks that take a mighty long time to complete. These tasks might involve fetchin' data from the depths of the internet, or performin' complex calculations that would make even Davy Jones scratch his head.

In the olden days, we pirates used to wait patiently for each task to finish afore movin' on to the next one. But that be a waste of precious time, me hearties! We be pirates, always lookin' for ways to be more efficient and plunder more booty!

That be where asynchronous programming comes in, me mateys. It be a way to tackle multiple tasks at once, without waitin' for each one to finish afore movin' on. It be like havin' a crew of scallywags workin' on different tasks simultaneously, while ye be overseein' the whole operation.

Ye see,

## 3. 为 gpt-3.5-turbo-0301 编写指令的技巧

为模型编写指令的最佳实践可能因模型版本而异。以下建议适用于 `gpt-3.5-turbo-0301`，可能不适用于未来的模型。

### System 消息

system 消息可用于为助手设定不同的个性或行为。

请注意，`gpt-3.5-turbo-0301` 对 system 消息的关注程度通常不如 `gpt-4-0314` 或 `gpt-3.5-turbo-0613`。因此，对于 `gpt-3.5-turbo-0301`，我们建议把重要的指令放在 user 消息中。一些开发者发现，随着对话变长，不断把 system 消息移到对话末尾附近，可以有效防止模型的注意力偏移。

In [12]:
# An example of a system message that primes the assistant to explain concepts in great depth
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a friendly and helpful teaching assistant. You explain concepts in great depth using simple terms, and you give examples to help people learn. At the end of each explanation, you ask a question to check for understanding"},
        {"role": "user", "content": "Can you explain how fractions work?"},
    ],
    temperature=0,
)

print(response.choices[0].message.content)


Of course! Fractions are a way to represent parts of a whole. They are made up of two numbers: a numerator and a denominator. The numerator tells you how many parts you have, and the denominator tells you how many equal parts make up the whole.

Let's take an example to understand this better. Imagine you have a pizza that is divided into 8 equal slices. If you eat 3 slices, you can represent that as the fraction 3/8. Here, the numerator is 3 because you ate 3 slices, and the denominator is 8 because the whole pizza is divided into 8 slices.

Fractions can also be used to represent numbers less than 1. For example, if you eat half of a pizza, you can write it as 1/2. Here, the numerator is 1 because you ate one slice, and the denominator is 2 because the whole pizza is divided into 2 equal parts.

Now, let's talk about equivalent fractions. Equivalent fractions are different fractions that represent the same amount. For example, 1/2 and 2/4 are equivalent fractions because they both re

In [13]:
# An example of a system message that primes the assistant to give brief, to-the-point answers
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a laconic assistant. You reply with brief, to-the-point answers with no elaboration."},
        {"role": "user", "content": "Can you explain how fractions work?"},
    ],
    temperature=0,
)

print(response.choices[0].message.content)


Fractions represent parts of a whole. They have a numerator (top number) and a denominator (bottom number).


### 少样本提示

在某些情况下，向模型展示你想要什么，比告诉模型你想要什么更容易。

向模型展示你想要什么的一种方法是使用伪造的示例消息。

例如：

In [14]:
# An example of a faked few-shot conversation to prime the model into translating business jargon to simpler speech
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful, pattern-following assistant."},
        {"role": "user", "content": "Help me translate the following corporate jargon into plain English."},
        {"role": "assistant", "content": "Sure, I'd be happy to!"},
        {"role": "user", "content": "New synergies will help drive top-line growth."},
        {"role": "assistant", "content": "Things working well together will increase revenue."},
        {"role": "user", "content": "Let's circle back when we have more bandwidth to touch base on opportunities for increased leverage."},
        {"role": "assistant", "content": "Let's talk later when we're less busy about how to do better."},
        {"role": "user", "content": "This late pivot means we don't have time to boil the ocean for the client deliverable."},
    ],
    temperature=0,
)

print(response.choices[0].message.content)


This sudden change in direction means we don't have enough time to complete the entire project for the client.


为了表明示例消息不属于真实对话、模型不应回头引用它们，你可以尝试将 `system` 消息的 `name` 字段设置为 `example_user` 和 `example_assistant`。

改写上面的少样本示例，可以这样写：

In [15]:
# The business jargon translation example, but with example names for the example messages
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful, pattern-following assistant that translates corporate jargon into plain English."},
        {"role": "system", "name":"example_user", "content": "New synergies will help drive top-line growth."},
        {"role": "system", "name": "example_assistant", "content": "Things working well together will increase revenue."},
        {"role": "system", "name":"example_user", "content": "Let's circle back when we have more bandwidth to touch base on opportunities for increased leverage."},
        {"role": "system", "name": "example_assistant", "content": "Let's talk later when we're less busy about how to do better."},
        {"role": "user", "content": "This late pivot means we don't have time to boil the ocean for the client deliverable."},
    ],
    temperature=0,
)

print(response.choices[0].message.content)


This sudden change in direction means we don't have enough time to complete the entire project for the client.


并非每次设计对话的尝试都能一次成功。

如果最初的尝试失败了，不要害怕尝试用不同的方式来引导或调节模型。

例如，有位开发者发现，插入一条内容为 "Great job so far, these have been perfect" 的 user 消息，有助于引导模型给出更高质量的回答，从而提高了准确率。

有关提升模型可靠性的更多思路，可以阅读我们的指南[提高可靠性的技巧](../articles/techniques_to_improve_reliability.md)。该指南是为非聊天模型编写的，但其中的许多原则仍然适用。

## 4. 计算 token 数量

提交请求时，API 会将消息转换为 token 序列。

所用的 token 数量会影响：
- 请求的成本
- 生成响应所需的时间
- 回复何时因达到最大 token 限制而被截断（`gpt-3.5-turbo` 为 4,096，`gpt-4` 为 8,192）

你可以使用以下函数来计算一组消息将使用的 token 数量。

请注意，从消息计算 token 的具体方式可能因模型而异。请把下面函数算出的数量视为估计值，而不是永久不变的保证。

特别地，使用可选 functions 输入的请求会在下面计算的估计值之外消耗额外的 token。

有关计算 token 的更多内容，请阅读[如何使用 tiktoken 计算 token 数量](How_to_count_tokens_with_tiktoken.ipynb)。

In [16]:
import tiktoken


def num_tokens_from_messages(messages, model="gpt-3.5-turbo-0613"):
    """Return the number of tokens used by a list of messages."""
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        print("Warning: model not found. Using cl100k_base encoding.")
        encoding = tiktoken.get_encoding("cl100k_base")
    if model in {
        "gpt-3.5-turbo-0613",
        "gpt-3.5-turbo-16k-0613",
        "gpt-4-0314",
        "gpt-4-32k-0314",
        "gpt-4-0613",
        "gpt-4-32k-0613",
        }:
        tokens_per_message = 3
        tokens_per_name = 1
    elif model == "gpt-3.5-turbo-0301":
        tokens_per_message = 4  # every message follows <|start|>{role/name}\n{content}<|end|>\n
        tokens_per_name = -1  # if there's a name, the role is omitted
    elif "gpt-3.5-turbo" in model:
        print("Warning: gpt-3.5-turbo may update over time. Returning num tokens assuming gpt-3.5-turbo-0613.")
        return num_tokens_from_messages(messages, model="gpt-3.5-turbo-0613")
    elif "gpt-4" in model:
        print("Warning: gpt-4 may update over time. Returning num tokens assuming gpt-4-0613.")
        return num_tokens_from_messages(messages, model="gpt-4-0613")
    else:
        raise NotImplementedError(
            f"""num_tokens_from_messages() is not implemented for model {model}."""
        )
    num_tokens = 0
    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))
            if key == "name":
                num_tokens += tokens_per_name
    num_tokens += 3  # every reply is primed with <|start|>assistant<|message|>
    return num_tokens


In [23]:
# let's verify the function above matches the OpenAI API response
example_messages = [
    {
        "role": "system",
        "content": "You are a helpful, pattern-following assistant that translates corporate jargon into plain English.",
    },
    {
        "role": "system",
        "name": "example_user",
        "content": "New synergies will help drive top-line growth.",
    },
    {
        "role": "system",
        "name": "example_assistant",
        "content": "Things working well together will increase revenue.",
    },
    {
        "role": "system",
        "name": "example_user",
        "content": "Let's circle back when we have more bandwidth to touch base on opportunities for increased leverage.",
    },
    {
        "role": "system",
        "name": "example_assistant",
        "content": "Let's talk later when we're less busy about how to do better.",
    },
    {
        "role": "user",
        "content": "This late pivot means we don't have time to boil the ocean for the client deliverable.",
    },
]

for model in [
    # "gpt-3.5-turbo-0301",
    # "gpt-4-0314",
    # "gpt-4-0613",
    "gpt-3.5-turbo-1106",
    "gpt-3.5-turbo",
    "gpt-4",
    "gpt-4-1106-preview",
    ]:
    print(model)
    # example token count from the function defined above
    print(f"{num_tokens_from_messages(example_messages, model)} prompt tokens counted by num_tokens_from_messages().")
    # example token count from the OpenAI API
    response = client.chat.completions.create(model=model,
    messages=example_messages,
    temperature=0,
    max_tokens=1)
    token = response.usage.prompt_tokens
    print(f'{token} prompt tokens counted by the OpenAI API.')
    print()


gpt-3.5-turbo-1106
129 prompt tokens counted by num_tokens_from_messages().
129 prompt tokens counted by the OpenAI API.

gpt-3.5-turbo
129 prompt tokens counted by num_tokens_from_messages().
129 prompt tokens counted by the OpenAI API.

gpt-4
129 prompt tokens counted by num_tokens_from_messages().
129 prompt tokens counted by the OpenAI API.

gpt-4-1106-preview
129 prompt tokens counted by num_tokens_from_messages().
129 prompt tokens counted by the OpenAI API.

